# Autotune — Servidor Ollama Remoto com GPU (Kaggle v2)

Notebook limpo — instala tudo do zero em cada sessão.

## Pré-requisitos
- Acelerador GPU activado nas definições do notebook
- Token ngrok configurado na célula 5

## Arquitetura
```
Backend / RAG → URL ngrok → Ollama no Kaggle (GPU)
```

# 1 - Instalar dependências e Ollama

In [3]:
!apt-get install -y zstd lshw pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok -q
!ls /usr/local/lib/ollama/

Reading package lists...
Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                                        0.1%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
cuda_v12	   libggml-base.so.0.0.0     libggml-cpu-sandybridge.so  vulkan
cuda_v13	   libggml-cpu-alderlake.so  libg

# 2 - Imports e funções auxiliares

In [4]:
import os
import time
import subprocess
import requests
from datetime import datetime
from pyngrok import ngrok, conf

MODELS_DIR = '/kaggle/working/ollama_models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['OLLAMA_MODELS'] = MODELS_DIR

print('Modelos em:', MODELS_DIR)


def stop_ollama():
    print('A parar processos antigos do Ollama...')
    subprocess.run(['killall', 'ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    print('Processos antigos terminados.')


def start_ollama():
    env = os.environ.copy()
    env['PATH'] += ':/usr/local/bin'
    env['OLLAMA_MODELS'] = MODELS_DIR
    env['OLLAMA_HOST'] = '0.0.0.0'
    env['OLLAMA_ORIGINS'] = '*'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['OLLAMA_GPU_LAYERS'] = '999'

    print('A iniciar servidor Ollama...')
    process = subprocess.Popen(
        ['/usr/local/bin/ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env
    )

    # Aguardar o Ollama iniciar
    for i in range(12):
        time.sleep(5)
        try:
            r = requests.get('http://localhost:11434/api/tags', timeout=3)
            if r.status_code == 200:
                print('Servidor Ollama iniciado.')
                return process
        except:
            print(f'A aguardar... ({(i+1)*5}s)')

    print('AVISO: Ollama demorou mais do esperado.')
    return process


def check_ollama_status():
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=10)
        if r.status_code == 200:
            print('Ollama está ativo.')
            return True
        print('Ollama respondeu com status:', r.status_code)
        return False
    except Exception as e:
        print('Erro ao contactar o Ollama:', e)
        return False


def pull_model(model_name):
    print(f'A descarregar modelo: {model_name}')
    result = subprocess.run(
        f'OLLAMA_MODELS={MODELS_DIR} ollama pull {model_name}',
        shell=True
    )
    if result.returncode == 0:
        print(f'Modelo {model_name} pronto.')
    else:
        print(f'Erro ao descarregar {model_name}.')


def list_models():
    print('Modelos disponíveis:')
    subprocess.run(f'OLLAMA_MODELS={MODELS_DIR} ollama list', shell=True)


def test_generation(model_name):
    print(f'A testar modelo: {model_name}')
    try:
        r = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': model_name,
                'messages': [{'role': 'user', 'content': 'Responde apenas: OK'}],
                'stream': False
            },
            timeout=120
        )
        if r.status_code == 200:
            print('Resposta:', r.json().get('message', {}).get('content', ''))
            print(f'{model_name} está a funcionar.')
        else:
            print('Erro:', r.text[:200])
    except Exception as e:
        print('Erro ao testar:', e)


def start_ngrok_tunnel(ngrok_token, domain, port=11434):
    conf.get_default().auth_token = ngrok_token
    print('A limpar túneis antigos...')
    ngrok.kill()
    print('A criar túnel ngrok...')
    tunnel = ngrok.connect(port, domain=domain)
    print('=' * 60)
    print('OLLAMA_RAG_BASE_URL=' + tunnel.public_url)
    print('=' * 60)
    return tunnel


def keep_alive(interval_seconds=30):
    print('Servidor ativo. Keep-alive a correr...')
    print('-' * 40)
    while True:
        try:
            r = requests.get('http://localhost:11434/api/tags', timeout=5)
            models = [m['name'] for m in r.json().get('models', [])]
            print('[' + datetime.now().strftime('%H:%M:%S') + '] OK - ' + ', '.join(models))
        except Exception as e:
            print('[' + datetime.now().strftime('%H:%M:%S') + '] ERRO: ' + str(e))
        time.sleep(interval_seconds)

Modelos em: /kaggle/working/ollama_models


# 3 - Iniciar Ollama e verificar GPU

In [5]:
stop_ollama()
ollama_process = start_ollama()
!nvidia-smi
check_ollama_status()

A parar processos antigos do Ollama...
Processos antigos terminados.
A iniciar servidor Ollama...
Servidor Ollama iniciado.
Fri May  8 11:42:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|               

True

# 4 - Descarregar modelos

In [6]:
pull_model('qwen2.5:14b')
list_models()

A descarregar modelo: qwen2.5:14b


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 2049f5674b1e:   0% ▕                  ▏  40 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏  87 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 181 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 287 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   4% ▕                  ▏ 340 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   5% ▕                  ▏ 438 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   6% ▕█                 ▏ 543 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b

Modelo qwen2.5:14b pronto.
Modelos disponíveis:
NAME           ID              SIZE      MODIFIED               
qwen2.5:14b    7cdf5a0187d5    9.0 GB    Less than a second ago    


pulling manifest 
pulling 2049f5674b1e: 100% ▕██████████████████▏ 9.0 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling db59b814cab7: 100% ▕██████████████████▏  488 B                         
verifying sha256 digest ⠇ pulling manifest 
pulling 2049f5674b1e: 100% ▕██████████████████▏ 9.0 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling db59b814cab7: 100% ▕██████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


# 5 - Testar modelos

In [7]:
test_generation('qwen2.5:14b')

A testar modelo: qwen2.5:14b
Resposta: OK
qwen2.5:14b está a funcionar.


# 6 - Configurar e iniciar ngrok

In [8]:
NGROK_TOKEN = '38MUdfuiCncbmMq2OrR0l9PqNTH_6Q6G7ugANjh51tDbqc2ZZ'
MEU_DOMINIO = 'irenic-daniel-unshowily.ngrok-free.dev'

ngrok.kill()
tunnel = start_ngrok_tunnel(
    ngrok_token=NGROK_TOKEN,
    domain=MEU_DOMINIO,
    port=11434
)

A limpar túneis antigos...
A criar túnel ngrok...
OLLAMA_RAG_BASE_URL=https://irenic-daniel-unshowily.ngrok-free.dev


# test_generation('qwen2.5:32b')7 - Keep-alive

In [ ]:
keep_alive(interval_seconds=30)

Servidor ativo. Keep-alive a correr...
----------------------------------------
[11:44:46] OK - qwen2.5:14b
[11:45:16] OK - qwen2.5:14b
[11:45:46] OK - qwen2.5:14b
[11:46:16] OK - qwen2.5:14b
[11:46:46] OK - qwen2.5:14b
[11:47:16] OK - qwen2.5:14b
[11:47:46] OK - qwen2.5:14b
[11:48:16] OK - qwen2.5:14b
[11:48:46] OK - qwen2.5:14b
[11:49:16] OK - qwen2.5:14b
[11:49:46] OK - qwen2.5:14b
[11:50:16] OK - qwen2.5:14b
[11:50:46] OK - qwen2.5:14b
[11:51:16] OK - qwen2.5:14b
[11:51:46] OK - qwen2.5:14b
[11:52:16] OK - qwen2.5:14b
[11:52:46] OK - qwen2.5:14b
[11:53:16] OK - qwen2.5:14b
[11:53:46] OK - qwen2.5:14b
[11:54:16] OK - qwen2.5:14b
[11:54:46] OK - qwen2.5:14b
[11:55:16] OK - qwen2.5:14b
[11:55:46] OK - qwen2.5:14b
[11:56:16] OK - qwen2.5:14b
[11:56:46] OK - qwen2.5:14b
[11:57:16] OK - qwen2.5:14b
[11:57:46] OK - qwen2.5:14b
[11:58:16] OK - qwen2.5:14b
[11:58:46] OK - qwen2.5:14b
[11:59:16] OK - qwen2.5:14b
[11:59:46] OK - qwen2.5:14b
[12:00:16] OK - qwen2.5:14b
[12:00:46] OK - qwen2.5: